In [1]:
import os
import sys

project_root = os.path.abspath("../../")
print(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

/home2/a/amiber/projects/phme26


In [2]:
from core.utils import load_best_params_lookup
import pandas as pd

/nfs/home/amiber/conda_envs/phme26_conda_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
def transform_results_to_df(output_data: dict) -> pd.DataFrame:
    """
    Flattens the nested training results into a clean Pandas DataFrame.
    """
    rows = []
    training_params = output_data.get("training_best_params", {})

    for dataset, windows in training_params.items():
        for win_id, details in windows.items():
            rows.append(
                {
                    "dataset": dataset,
                    "win_len": details.get("window_len"),
                    "best_lr": details.get("lr"),
                    "best_batch_size": details.get("batch_size"),
                    "best_loss_value": details.get("best_val_loss"),
                }
            )

    # Create the DataFrame
    df = pd.DataFrame(rows)

    # Optional: Sort by dataset and window length for better readability
    if not df.empty:
        df = df.sort_values(by=["dataset", "win_len"]).reset_index(drop=True)

    return df


In [13]:
def get_best_per_dataset(df):
    # 1. Group by dataset
    # 2. Find the index (idxmin) of the row with the minimum loss for each group
    # 3. Use .loc to pull the full rows at those indices
    best_rows = df.loc[df.groupby("dataset")["best_loss_value"].idxmin()]

    return best_rows.sort_values("dataset")


In [16]:
diction_raw = load_best_params_lookup("best_params_raw_scaled_sequenced_lstm.jsonl")
diction_raw_env = load_best_params_lookup("best_params_raw_env_scaled_sequenced_lstm.jsonl")

In [18]:
df_raw = transform_results_to_df(diction_raw["Raw -> Scaled -> Sequenced -> LSTM"])
df_raw

,dataset,win_len,best_lr,best_batch_size,best_loss_value
0,cwru,10,0.00050,8,3.876131e-02
1,cwru,20,0.00050,16,3.800455e-02
2,cwru,100,0.00050,32,1.730367e-01
3,cwru,200,0.00050,8,2.449051e-01
4,cwru,400,0.00005,128,3.858986e-01
5,cwru,800,0.00005,64,4.592713e-01
6,cwru,1200,0.00050,128,5.008727e-01
7,cwru,2400,0.00005,64,6.222520e-01
8,kaist,10,0.00050,8,1.777330e-04
9,kaist,20,0.00050,8,4.293582e-05


In [19]:
winners_df_raw = get_best_per_dataset(df_raw)
winners_df_raw

,dataset,win_len,best_lr,best_batch_size,best_loss_value
1,cwru,20,0.0005,16,3.800455e-02
10,kaist,100,0.0005,8,2.207579e-09
17,mfpt,20,0.0005,16,2.506907e-02


In [21]:
df_raw_env = transform_results_to_df(diction_raw_env["Raw -> Env -> Scaled -> Sequenced -> LSTM"])
df_raw_env

,dataset,win_len,best_lr,best_batch_size,best_loss_value
0,cwru,10,0.00050,8,0.054122
1,cwru,20,0.00050,8,0.006272
2,cwru,100,0.00050,16,0.102647
3,cwru,200,0.00050,8,0.198216
4,cwru,400,0.00005,32,0.200902
5,cwru,800,0.00005,64,0.333064
6,cwru,1200,0.00010,128,0.374934
7,cwru,2400,0.00050,8,0.534207
8,kaist,10,0.00050,32,0.006585
9,kaist,20,0.00050,16,0.000112


In [22]:
winners_df_raw_env = get_best_per_dataset(df_raw_env)
winners_df_raw_env

,dataset,win_len,best_lr,best_batch_size,best_loss_value
1,cwru,20,0.0005,8,0.006272
9,kaist,20,0.0005,16,0.000112
23,mfpt,2400,0.0001,8,0.047739
